In [11]:
import pandas as pd
import os

In [ ]:
BASE_DIR = "../data"

# skiprows=4 skips the World Bank metadata header; drop redundant/artifact columns
education_spend_df = pd.read_csv(
    f"{BASE_DIR}/education-spend/country-education-spend.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

gini_df = pd.read_csv(
    f"{BASE_DIR}/GINI-index/country-GINI-index.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

corruption_df = pd.read_csv(
    f"{BASE_DIR}/corruption_index/country-corruption-index.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

regulation_df = pd.read_csv(
    f"{BASE_DIR}/regulation/CPIA.csv",
    skiprows = 4
).drop(columns=["Indicator Name", "Indicator Code", "Country Code", "Unnamed: 70"])

# Unlike the other two sources, this file has every country row duplicated verbatim; drop the
# duplicates so corruption_df lines up 1:1 with gini_df/education_spend_df like the other two do
corruption_df = corruption_df.drop_duplicates().reset_index(drop=True)

# Nauru is listed under its Nauruan name ("Naoero") here but as "Nauru" in the other two
# datasets -- align it so it isn't silently dropped from the later merge on "Country Name"
corruption_df["Country Name"] = corruption_df["Country Name"].replace("Naoero", "Nauru")

# Same Nauru/"Naoero" naming quirk shows up in this source too
regulation_df["Country Name"] = regulation_df["Country Name"].replace("Naoero", "Nauru")

In [ ]:
# Year columns are strings; keep only non-year columns and years >= 2000
cols_to_drop = [col for col in gini_df.columns if str(col).strip().isdigit() and int(col) < 2000]
gini_df = gini_df.drop(columns=cols_to_drop)
education_spend_df = education_spend_df.drop(columns=cols_to_drop)
corruption_df = corruption_df.drop(columns=cols_to_drop)
regulation_df = regulation_df.drop(columns=cols_to_drop)

In [ ]:
# sort_values() returns a copy; without reassignment, these two calls don't actually sort anything
gini_df.sort_values("Country Name")
education_spend_df.sort_values("Country Name")
corruption_df.sort_values("Country Name")
regulation_df.sort_values("Country Name")

number_of_rows_gini = list(gini_df.shape)

# Flag rows where every year column is empty (NaN, or a string like "", "nan", "None")
empty_mask = gini_df.loc[:, gini_df.columns[1:]].apply(
    lambda col: col.astype(str).str.strip().isin(["", "nan", "None"]) | col.isna()
).all(axis=1)

# Drop those rows from both dataframes together so they stay row-aligned for the later merge
gini_df = gini_df[~empty_mask]
education_spend_df = education_spend_df[~empty_mask]
corruption_df = corruption_df[~empty_mask]
regulation_df = regulation_df[~empty_mask]

# Same check, but for rows with no education-spend data
empty_mask_2 = education_spend_df.loc[:, education_spend_df.columns[1:]].apply(
    lambda col: col.astype(str).str.strip().isin(["", "nan", "None"]) | col.isna()
).all(axis=1)

gini_df = gini_df[~empty_mask_2]
education_spend_df = education_spend_df[~empty_mask_2]
corruption_df = corruption_df[~empty_mask_2]
regulation_df = regulation_df[~empty_mask_2]

empty_mask_3 = corruption_df.loc[:, corruption_df.columns[1:]].apply(
    lambda col: col.astype(str).str.strip().isin(["", "nan", "None"]) | col.isna()
).all(axis=1)

gini_df = gini_df[~empty_mask_3]
education_spend_df = education_spend_df[~empty_mask_3]
corruption_df = corruption_df[~empty_mask_3]
regulation_df = regulation_df[~empty_mask_3]

# Same check, but for rows with no regulation data
empty_mask_4 = regulation_df.loc[:, regulation_df.columns[1:]].apply(
    lambda col: col.astype(str).str.strip().isin(["", "nan", "None"]) | col.isna()
).all(axis=1)

gini_df = gini_df[~empty_mask_4]
education_spend_df = education_spend_df[~empty_mask_4]
corruption_df = corruption_df[~empty_mask_4]
regulation_df = regulation_df[~empty_mask_4]

gini_df

In [ ]:
year_cols_numeric = [c for c in gini_df.columns if c != "Country Name"]
gini_df[year_cols_numeric] = gini_df[year_cols_numeric].round(3)
education_spend_df[year_cols_numeric] = education_spend_df[year_cols_numeric].round(3)
corruption_df[year_cols_numeric] = corruption_df[year_cols_numeric].round(3)
regulation_df[year_cols_numeric] = regulation_df[year_cols_numeric].round(3)

education_spend_df

In [6]:
corruption_df

,Country Name,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
4,Angola,21.937,NaN,21.457,21.072,23.102,21.661,21.606,21.937,22.150,...,17.731,19.095,23.934,28.326,29.803,34.867,35.646,35.219,35.253,NaN
5,Albania,27.962,NaN,27.606,27.452,32.564,29.981,33.792,34.188,34.823,...,37.609,36.391,35.971,36.955,34.507,35.669,37.725,38.462,39.338,NaN
8,United Arab Emirates,52.640,NaN,63.458,65.802,63.541,63.186,63.677,63.602,66.321,...,70.251,69.639,70.015,69.874,69.340,68.804,68.549,67.932,71.542,NaN
9,Argentina,41.679,NaN,36.476,36.758,39.647,39.979,41.360,39.671,39.479,...,43.415,44.249,47.739,46.140,45.133,41.229,40.968,40.688,41.262,NaN
10,Armenia,31.163,NaN,28.827,28.799,34.249,35.735,35.127,33.572,32.479,...,36.101,36.436,44.530,46.795,49.992,48.985,49.019,49.972,50.783,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,Samoa,46.660,NaN,46.473,46.409,44.447,53.359,53.472,53.391,53.413,...,51.675,60.320,60.373,60.446,64.269,63.375,63.363,64.430,61.783,NaN
261,"Yemen, Rep.",29.471,NaN,26.192,25.313,24.272,26.061,31.513,30.912,32.334,...,15.264,15.651,12.371,12.600,15.474,15.629,14.127,15.426,14.742,NaN
262,South Africa,57.423,NaN,51.927,50.637,52.323,54.714,57.154,54.092,51.085,...,48.475,46.238,46.897,48.563,48.374,49.162,44.546,43.612,43.973,NaN
263,Zambia,29.084,NaN,31.625,31.472,35.597,35.630,39.919,40.827,40.853,...,39.276,38.462,35.375,34.787,35.607,33.830,39.392,39.796,38.608,NaN


In [ ]:
# how="right" keeps every row in gini_df even when a country has no education-spend match;
# suffixes disambiguate the duplicate year columns coming from each source
combined = pd.merge(education_spend_df, gini_df, on="Country Name", how="right", suffixes=("_edu", "_gini"))
combined = pd.merge(combined, corruption_df, on="Country Name", how="left")
combined = pd.merge(combined, regulation_df, on="Country Name", how="left", suffixes=("", "_reg"))

year_cols = [col for col in gini_df.columns if col != "Country Name"]

# Collapse each year's _gini/_edu/corruption/regulation columns into a single
# [gini, education_spend, corruption, regulation] quadruple; NaN if any value is missing,
# so downstream filtering only needs one check
for year in year_cols:
    quadruples = combined[[f"{year}_gini", f"{year}_edu", year, f"{year}_reg"]].values.tolist()
    combined[year] = [np.nan if any(pd.isna(v) for v in quadruple) else quadruple for quadruple in quadruples]
    combined = combined.drop(columns=[f"{year}_gini", f"{year}_edu", f"{year}_reg"])

combined

In [ ]:
combined.to_csv("../data/processed/combined_cleaned.csv", index=False)
gini_df.to_csv("../data/processed/gini_cleaned.csv", index=False)
education_spend_df.to_csv("../data/processed/education_cleaned.csv", index=False)
corruption_df.to_csv("../data/processed/corruption_cleaned.csv", index=False)
regulation_df.to_csv("../data/processed/regulation_cleaned.csv", index=False)

In [ ]:
# Reshape from wide (one column per year) to long (one row per country-year) format
melted = combined.melt(id_vars="Country Name", var_name="Year", value_name="Values")

# Keep only rows where a real [gini, edu, corruption, regulation] quadruple survived (drops the NaN placeholders)
melted = melted[melted["Values"].apply(lambda v: isinstance(v, list))].copy()

# NOTE: this overwrites the "Year" column melt() already set correctly from the original column
# names, renumbering each country's remaining rows sequentially from 2000 via cumcount(). That's
# only correct if a country's data has no gaps -- any missing year shifts every later year off by one.
melted["Year"] = melted.groupby("Country Name").cumcount() + 2000

melted["GINI"] = melted["Values"].apply(lambda v: v[0])
melted["Education_Spend"] = melted["Values"].apply(lambda v: v[1])
melted["Corruption"] = melted["Values"].apply(lambda v: v[2])
melted["Regulation"] = melted["Values"].apply(lambda v: v[3])

reshaped_GINI_education_df = melted[["Country Name", "Year", "GINI", "Education_Spend", "Corruption", "Regulation"]].rename(
    columns={"Country Name": "Country"}
).reset_index(drop=True)

reshaped_GINI_education_df.sort_values("Country")

In [10]:
reshaped_GINI_education_df.to_csv("../data/processed/long_form.csv")